In [120]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
import warnings
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import train_test_split
import warnings
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

In [ ]:
df_outer = pd.read_excel('data/test.xlsx')
df_inner = pd.read_excel('data/train.xlsx')
df_inner['group'] = 'inner'
df_outer['group'] = 'outer'

In [ ]:
df = pd.concat([df_inner, df_outer])
df = df.reset_index(drop=True)

In [ ]:
def f1(x):
    if x == 'NCR':
        return 0 
    elif x == 'CR':
        return 1
    else:
        return -1
df.label = df.label.map(f1)

In [ ]:
df.label.value_counts()

In [ ]:
tableFeats = pd.read_csv('data/featName_table.csv').featName.tolist()  

In [ ]:
tb = pd.read_csv('results/LASSO_Table.csv')   
selected_features = tb[tb.coef!=0].Feature.values   

In [ ]:
text1Feats = pd.read_csv('data/featName_text1.csv').featName.tolist()

In [ ]:
tb = pd.read_csv('results/LASSO_Text1.csv')
selected_features_text1 = tb[tb.coef!=0].Feature.values

In [ ]:
text2Feats = pd.read_csv('data/featName_text2.csv').featName.tolist()

In [ ]:
tb = pd.read_csv('results/LASSO_Text2.csv')
selected_features_text2 = tb[tb.coef!=0].Feature.values

In [ ]:
X = df[tableFeats+text1Feats+text2Feats].copy()
X = X[list(selected_features)+list(selected_features_text1)+list(selected_features_text2)]
print(X.shape)

In [ ]:
outer_index = df[df.group=='outer'].index.values
X_outer = X.iloc[outer_index, :].copy()

inner_index = df[df.group=='inner'].index.values
X_inner = X.iloc[inner_index, :].copy()

In [ ]:
y = df.label.values
y_outer = y[outer_index].copy()
y_inner = y[inner_index].copy()

In [ ]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score, f1_score,
    precision_score, confusion_matrix, average_precision_score, brier_score_loss
)
import numpy as np
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=218)
best_params = {'n_estimators': 141, 'max_depth': 26, 'learning_rate': 0.09730417648170471, 'subsample': 0.9459857157072243, 'colsample_bytree': 0.5440944072548214, 'gamma': 1.756504445267348, 'reg_alpha': 0.6154914567689445, 'reg_lambda': 1.1596104320505658}
train_metrics = []
test_metrics = []

y_true_test_all, y_pred_test_all, y_proba_test_all = [], [], []
y_true_train_all, y_pred_train_all, y_proba_train_all = [], [], []

for train_idx, test_idx in cv.split(X_inner, y_inner):
    X_train, X_test = X.values[train_idx], X.values[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    weights = compute_sample_weight(class_weight='balanced', y=y_train)

    model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train, sample_weight=weights)

    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_true_train_all.extend(y_train)
    y_pred_train_all.extend(y_pred_train)
    y_proba_train_all.extend(y_proba_train)

    y_pred_test = model.predict(X_test)
    y_proba_test = model.predict_proba(X_test)[:, 1]
    y_true_test_all.extend(y_test)
    y_pred_test_all.extend(y_pred_test)
    y_proba_test_all.extend(y_proba_test)

    def get_metrics(y_true, y_pred, y_proba):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        return {
            'auc': roc_auc_score(y_true, y_proba),
            'auc_pr': average_precision_score(y_true, y_proba),  
            'brier': brier_score_loss(y_true, y_proba),         
            'acc': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred),
            'sensitivity': sensitivity,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv
        }

    train_metrics.append(get_metrics(y_train, y_pred_train, y_proba_train))
    test_metrics.append(get_metrics(y_test, y_pred_test, y_proba_test))

def evaluate_cv_metrics(metrics_list, prefix=''):
    print(f'\n{prefix} Cross-Validation Mean ± Std:')
    for key in metrics_list[0].keys():
        values = [m[key] for m in metrics_list]
        mean = np.mean(values)
        std = np.std(values)
        print(f"{key.capitalize():<12}: {mean:.4f} ± {std:.4f}")

evaluate_cv_metrics(train_metrics, prefix='Train')
evaluate_cv_metrics(test_metrics, prefix='Test')

weights = compute_sample_weight(class_weight='balanced', y=y_inner)
model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_inner, y_inner, sample_weight=weights)
y_pred_outer = model.predict(X_outer)
y_proba_outer = model.predict_proba(X_outer)[:, 1]

def evaluate(y_true, y_pred, y_proba, prefix=''):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    auc = roc_auc_score(y_true, y_proba)
    auc_pr = average_precision_score(y_true, y_proba)  
    brier = brier_score_loss(y_true, y_proba)         
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{prefix} Metrics:')
    print(f"AUC:         {auc:.4f}")
    print(f"AUC-PR:      {auc_pr:.4f}")  
    print(f"Brier:       {brier:.4f}")   
    print(f"Accuracy:    {acc:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score:    {f1:.4f}")
    print(f"PPV:         {ppv:.4f}")
    print(f"NPV:         {npv:.4f}")

evaluate(y_outer, y_pred_outer, y_proba_outer, prefix='Outer Test (External)')